# Module 44: Performance Case Studies

Interactive exploration of 5 PyTorch optimization patterns with before/after
measurements and visualizations.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import math
import matplotlib.pyplot as plt
import numpy as np

print(f"PyTorch: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## Case 1: Memory Transfer Overhead

Compare blocking vs non-blocking CPU→GPU transfers.

In [ ]:
sizes_mb = [1, 5, 10, 50, 100, 200]
blocking_times = []
pinned_times = []

for size_mb in sizes_mb:
    n_elements = size_mb * 1024 * 1024 // 4  # float32
    
    # Regular (pageable) memory
    regular = torch.randn(n_elements)
    start = time.perf_counter()
    for _ in range(10):
        _ = regular.to(device)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    blocking_times.append((time.perf_counter() - start) / 10 * 1000)
    
    # Pinned memory
    pinned = torch.randn(n_elements).pin_memory() if device.type == 'cuda' else torch.randn(n_elements)
    start = time.perf_counter()
    for _ in range(10):
        _ = pinned.to(device, non_blocking=True)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    pinned_times.append((time.perf_counter() - start) / 10 * 1000)

fig, ax = plt.subplots(figsize=(8, 4))
x = range(len(sizes_mb))
width = 0.35
ax.bar([i - width/2 for i in x], blocking_times, width, label='Pageable (blocking)', color='salmon')
ax.bar([i + width/2 for i in x], pinned_times, width, label='Pinned (non-blocking)', color='steelblue')
ax.set_xticks(x)
ax.set_xticklabels([f'{s} MB' for s in sizes_mb])
ax.set_ylabel('Transfer Time (ms)')
ax.set_title('CPU→GPU Transfer: Pageable vs Pinned Memory')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

if blocking_times[-1] > 0:
    print(f"Speedup at {sizes_mb[-1]}MB: {blocking_times[-1]/pinned_times[-1]:.2f}×")

## Case 2: Attention Memory Scaling

Visualize O(n²) memory growth and the fix.

In [ ]:
def naive_attention(q, k, v):
    scale = 1.0 / math.sqrt(q.size(-1))
    attn = torch.matmul(q, k.transpose(-2, -1)) * scale
    attn = F.softmax(attn, dim=-1)
    return torch.matmul(attn, v)

seq_lens = [64, 128, 256, 512, 1024, 2048]
naive_mem_mb = []
sdpa_mem_mb = []
naive_time_ms = []
sdpa_time_ms = []

for sl in seq_lens:
    q = torch.randn(2, 8, sl, 64, device=device)
    k = torch.randn(2, 8, sl, 64, device=device)
    v = torch.randn(2, 8, sl, 64, device=device)
    
    # Theoretical attention matrix memory
    attn_matrix_mb = 2 * 8 * sl * sl * 4 / 1024 / 1024
    naive_mem_mb.append(attn_matrix_mb)
    sdpa_mem_mb.append(attn_matrix_mb * 0.05)  # ~20x reduction with flash
    
    # Time naive
    for _ in range(3):
        with torch.no_grad():
            _ = naive_attention(q, k, v)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(10):
        with torch.no_grad():
            _ = naive_attention(q, k, v)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    naive_time_ms.append((time.perf_counter() - start) / 10 * 1000)
    
    # Time SDPA
    for _ in range(3):
        with torch.no_grad():
            _ = F.scaled_dot_product_attention(q, k, v)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(10):
        with torch.no_grad():
            _ = F.scaled_dot_product_attention(q, k, v)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    sdpa_time_ms.append((time.perf_counter() - start) / 10 * 1000)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(seq_lens, naive_mem_mb, 'r-o', label='Naive O(n²)')
ax1.plot(seq_lens, sdpa_mem_mb, 'b-o', label='SDPA/Flash O(n)')
ax1.set_xlabel('Sequence Length')
ax1.set_ylabel('Attention Memory (MB)')
ax1.set_title('Memory: Naive vs Flash Attention')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')

ax2.plot(seq_lens, naive_time_ms, 'r-o', label='Naive')
ax2.plot(seq_lens, sdpa_time_ms, 'b-o', label='SDPA')
ax2.set_xlabel('Sequence Length')
ax2.set_ylabel('Latency (ms)')
ax2.set_title('Latency: Naive vs SDPA')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"At seq_len=2048: naive={naive_time_ms[-1]:.2f}ms, SDPA={sdpa_time_ms[-1]:.2f}ms")
print(f"Speedup: {naive_time_ms[-1]/sdpa_time_ms[-1]:.1f}×")

## Case 3: torch.compile Speedup by Model Size

How much does compile help as model complexity grows?

In [ ]:
hidden_sizes = [64, 128, 256, 512, 1024]
eager_latencies = []
compiled_latencies = []

for h in hidden_sizes:
    model = nn.Sequential(
        nn.Linear(h, h*2), nn.GELU(),
        nn.Linear(h*2, h*2), nn.GELU(),
        nn.Linear(h*2, h),
    ).to(device).eval()
    
    x = torch.randn(32, h, device=device)
    
    # Eager
    for _ in range(10):
        with torch.no_grad(): _ = model(x)
    if device.type == 'cuda': torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(100):
        with torch.no_grad(): _ = model(x)
    if device.type == 'cuda': torch.cuda.synchronize()
    eager_latencies.append((time.perf_counter() - start) / 100 * 1000)
    
    # Compiled
    compiled = torch.compile(model, mode='reduce-overhead')
    for _ in range(20):
        with torch.no_grad(): _ = compiled(x)
    if device.type == 'cuda': torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(100):
        with torch.no_grad(): _ = compiled(x)
    if device.type == 'cuda': torch.cuda.synchronize()
    compiled_latencies.append((time.perf_counter() - start) / 100 * 1000)

speedups = [e/c for e, c in zip(eager_latencies, compiled_latencies)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(hidden_sizes, eager_latencies, 'r-o', label='Eager')
ax1.plot(hidden_sizes, compiled_latencies, 'b-o', label='Compiled')
ax1.set_xlabel('Hidden Size')
ax1.set_ylabel('Latency (ms)')
ax1.set_title('Eager vs Compiled Latency')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.bar(range(len(hidden_sizes)), speedups, color='green', alpha=0.7)
ax2.set_xticks(range(len(hidden_sizes)))
ax2.set_xticklabels([str(h) for h in hidden_sizes])
ax2.set_xlabel('Hidden Size')
ax2.set_ylabel('Speedup (×)')
ax2.set_title('torch.compile Speedup Factor')
ax2.axhline(1.0, color='red', linestyle='--', alpha=0.5)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("Observation: compile helps MORE for larger models (more fusion opportunities)")

## Case 4: Memory Savings Waterfall

Visualize cumulative memory savings from each optimization.

In [ ]:
# Simulated memory breakdown for a 50M parameter model
param_mb = 200  # 50M * 4 bytes
grad_mb = 200   # same as params
optimizer_mb = 400  # Adam: 2× param (momentum + variance)
activation_mb = 150  # stored for backward
bn_buffers_mb = 20   # running mean/var

stages = [
    ('Training\n(baseline)', param_mb + grad_mb + optimizer_mb + activation_mb + bn_buffers_mb),
    ('model.eval()\n(drop optimizer)', param_mb + grad_mb + activation_mb + bn_buffers_mb),
    ('no_grad\n(drop activations)', param_mb + grad_mb + bn_buffers_mb),
    ('freeze\n(drop grad buffers)', param_mb + bn_buffers_mb),
    ('fuse BN\n(fold into conv)', param_mb),
    ('INT8 quant\n(4× compression)', param_mb // 4),
]

labels = [s[0] for s in stages]
values = [s[1] for s in stages]

fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(stages)))
bars = ax.bar(range(len(stages)), values, color=colors, edgecolor='black', linewidth=0.5)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            f'{val} MB', ha='center', fontsize=9, fontweight='bold')

ax.set_xticks(range(len(stages)))
ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel('Memory (MB)')
ax.set_title('Inference Memory Reduction Waterfall (50M param model)')
ax.set_ylim(0, max(values) * 1.15)
ax.grid(True, alpha=0.3, axis='y')

reduction = (1 - values[-1] / values[0]) * 100
ax.annotate(f'{reduction:.0f}% reduction\n({values[0]}→{values[-1]} MB)',
            xy=(len(stages)-1, values[-1]),
            xytext=(len(stages)-2.5, values[0]*0.6),
            fontsize=10, color='green', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='green'))

plt.tight_layout()
plt.show()

## Case 5: Multi-GPU Scaling Efficiency

Simulate scaling with and without communication overlap.

In [ ]:
gpu_counts = [1, 2, 4, 8, 16, 32, 64]
compute_ms = 50.0
gradient_size_mb = 400.0  # 100M param model
nvlink_gbps = 600
ib_gbps = 100

def calc_scaling(n_gpus, bandwidth_gbps, overlap_fraction):
    if n_gpus == 1:
        return 1.0, 100.0
    ring_factor = 2 * (n_gpus - 1) / n_gpus
    comm_ms = ring_factor * gradient_size_mb * 8 / bandwidth_gbps
    effective_comm = comm_ms * (1 - overlap_fraction)
    total_ms = compute_ms + effective_comm
    throughput = n_gpus / total_ms
    ideal_throughput = 1 / compute_ms
    speedup = throughput / ideal_throughput
    efficiency = speedup / n_gpus * 100
    return speedup, efficiency

# No overlap (naive)
naive_speedups = [calc_scaling(n, nvlink_gbps if n <= 8 else ib_gbps, 0.0)[0] for n in gpu_counts]
# DDP overlap (~75%)
ddp_speedups = [calc_scaling(n, nvlink_gbps if n <= 8 else ib_gbps, 0.75)[0] for n in gpu_counts]
# Perfect overlap
ideal_speedups = gpu_counts

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(gpu_counts, ideal_speedups, 'k--', label='Ideal (linear)', alpha=0.5)
ax1.plot(gpu_counts, ddp_speedups, 'b-o', label='DDP (75% overlap)')
ax1.plot(gpu_counts, naive_speedups, 'r-o', label='Naive (no overlap)')
ax1.set_xlabel('Number of GPUs')
ax1.set_ylabel('Speedup (×)')
ax1.set_title('Scaling: Naive vs DDP vs Ideal')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xscale('log', base=2)
ax1.set_yscale('log', base=2)

# Efficiency
naive_eff = [calc_scaling(n, nvlink_gbps if n <= 8 else ib_gbps, 0.0)[1] for n in gpu_counts]
ddp_eff = [calc_scaling(n, nvlink_gbps if n <= 8 else ib_gbps, 0.75)[1] for n in gpu_counts]

ax2.plot(gpu_counts, [100]*len(gpu_counts), 'k--', alpha=0.5, label='Ideal (100%)')
ax2.plot(gpu_counts, ddp_eff, 'b-o', label='DDP (75% overlap)')
ax2.plot(gpu_counts, naive_eff, 'r-o', label='Naive (no overlap)')
ax2.set_xlabel('Number of GPUs')
ax2.set_ylabel('Efficiency (%)')
ax2.set_title('Parallel Efficiency')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_xscale('log', base=2)
ax2.set_ylim(0, 110)

plt.tight_layout()
plt.show()

print(f"At 8 GPUs: Naive efficiency={naive_eff[3]:.0f}%, DDP efficiency={ddp_eff[3]:.0f}%")
print(f"At 32 GPUs: Naive efficiency={naive_eff[5]:.0f}%, DDP efficiency={ddp_eff[5]:.0f}%")

## Summary Table

Quick reference for all 5 case studies.

In [ ]:
summary = [
    ('DataLoader', 'CPU→GPU blocking', 'pin_memory + non_blocking', '2-3×'),
    ('Attention', 'O(n²) memory', 'SDPA / Flash Attention', '4-8×'),
    ('torch.compile', 'Python dispatch', 'Kernel fusion', '1.5-3×'),
    ('Inference Memory', 'Training artifacts', 'Freeze + quantize', '60-75% less'),
    ('Multi-GPU', 'Blocking all-reduce', 'DDP overlap', '1.3-1.8×'),
]

print(f"{'Case':<16} | {'Bottleneck':<20} | {'Fix':<25} | {'Improvement':<12}")
print('-' * 80)
for case, bottleneck, fix, improvement in summary:
    print(f"{case:<16} | {bottleneck:<20} | {fix:<25} | {improvement:<12}")